In [7]:
# Imports and Configuration

import os
import time
import logging
from dataclasses import dataclass, field
from typing import Optional, List, Dict
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

MAX_RETRIES = 3
RETRY_DELAY = 2
DEFAULT_MODEL = "llama-3.3-70b-versatile"
DEFAULT_MAX_TOKENS = 1024
DEFAULT_TEMPERATURE = 0.7

In [8]:
# Client Initialization

env_path = Path("C:/educational files/advanced_agent/.env")
load_dotenv(dotenv_path=env_path)

def init_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY not found in .env")
    log.info("Groq client initialized successfully")
    return Groq(api_key=api_key)

client = init_client()

2026-06-02 11:10:56,729 [INFO] Groq client initialized successfully


In [9]:
# Agent Configuration
@dataclass
class AgentConfig:
    model: str = DEFAULT_MODEL
    max_tokens: int = DEFAULT_MAX_TOKENS
    temperature: float = DEFAULT_TEMPERATURE
    system_prompt: str = (
        "You are an advanced autonomous AI agent. "
        "You reason carefully, give structured responses, "
        "and maintain context across the entire conversation. "
        "Be concise, precise, and professional."
    )
    session_token_count: int = field(default=0, repr=False)

config = AgentConfig()
log.info(f"Agent configured — model: {config.model}, max_tokens: {config.max_tokens}")

2026-06-02 11:11:18,323 [INFO] Agent configured — model: llama-3.3-70b-versatile, max_tokens: 1024


In [10]:
# Conversation Manager

class ConversationManager:
    VALID_ROLES = {"user", "assistant"}

    def __init__(self):
        self.history: List[Dict[str, str]] = []

    def add(self, role: str, content: str) -> None:
        if role not in self.VALID_ROLES:
            raise ValueError(f"Invalid role: {role}")
        self.history.append({"role": role, "content": content})

    def clear(self) -> None:
        self.history.clear()
        log.info("Conversation history cleared")

    def get(self) -> List[Dict[str, str]]:
        return self.history.copy()

    def summary(self) -> str:
        return f"Turns: {len(self.history) // 2} | Messages: {len(self.history)}"

conversation = ConversationManager()
log.info("Conversation manager ready")

2026-06-02 11:11:26,142 [INFO] Conversation manager ready


In [11]:
# Chat Engine

def chat(user_input: str, manager: ConversationManager, cfg: AgentConfig) -> Optional[str]:
    manager.add("user", user_input)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=cfg.model,
                messages=[{"role": "system", "content": cfg.system_prompt}] + manager.get(),
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content
            manager.add("assistant", reply)
            tokens = response.usage.total_tokens
            cfg.session_token_count += tokens
            log.info(f"Tokens this turn: {tokens} | Session total: {cfg.session_token_count}")
            return reply

        except Exception as e:
            log.warning(f"Attempt {attempt + 1} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)

    log.error("All retry attempts failed")
    return None

In [12]:
# Interactive Chat Loop

print("Agent ready. Commands: 'exit' to quit | 'clear' to reset history | 'usage' for token count\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() == "exit":
        print(f"Session ended. Total tokens used: {config.session_token_count}")
        break
    if user_input.lower() == "clear":
        conversation.clear()
        print("History cleared.\n")
        continue
    if user_input.lower() == "usage":
        print(f"Session tokens: {config.session_token_count} | {conversation.summary()}\n")
        continue

    reply = chat(user_input, conversation, config)
    print(f"\nAgent: {reply}\n")

Agent ready. Commands: 'exit' to quit | 'clear' to reset history | 'usage' for token count



You:  hi


2026-06-02 11:12:04,488 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 11:12:04,537 [INFO] Tokens this turn: 95 | Session total: 95



Agent: Hello. It's nice to meet you. Is there something I can help you with or would you like to start a conversation?



You:  my name is charan


...

You:  what is my name


2026-06-02 11:12:40,659 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 11:12:40,676 [INFO] Tokens this turn: 176 | Session total: 419



Agent: Your name is Charan. We established that earlier in our conversation.



You:  exit


Session ended. Total tokens used: 419
